# QuackWeave stabilization with SQLLogicTest

Use DuckDB’s native SQLLogicTest runner for the unified native package. The same case manifest covers debug and release builds with SQL/PGQ `GRAPH_TABLE`, typed Cypher and GQL, and standalone graph reads. The release package also runs the observer audit.

| Run | Case groups | SQLLogic blocks |
|---|---:|---:|
| Native debug | 135 | 657 |
| Native release | 135 | 657 |
| Native observer | 7 | 25 |

The macOS ARM64 verification covered 142 distinct case groups and 277 case/run pairs. Counts exclude the DuckDB version probes. The runner and library use DuckDB `v2.0.1`, source `8d78114e90`, with Ladybug `v0.22.3`.

[Capability and evidence guide](docs/index.html) · [Case manifest](test/stabilization/case_manifest.csv) · [Fixture launcher](test/stabilization/run_sqllogictest.py)

## DuckDB code and conventions

I inspected the registered `duckdb` project with `knowledge_context_pack_2`, `code_symbol_search`, and `code_read_symbol`. The worktree is newer than the product ABI pin. The indexed `result_helper.cpp` blob `c66a885c17abe4874a3dac480517d390f7007697` matches `v2.0.1`. The command and runner blobs differ, so I used the pinned source and tested the relevant conventions in the exact-version runner.

| Symbol or example | Design decision |
|---|---|
| `SQLLogicTestRunner::ExecuteFile` and `LoadDatabase` | Use a fresh DuckDB instance per file. Pass fixture paths with `test-env` and `--env-passthrough`. Use named connections when a cross-connection config change or recovery matters. [Runner source](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/test/sqlite/sqllogic_test_runner.cpp) |
| `Command::ExecuteQuery` and `TestResultHelper::CheckQueryResult` | Let the runner compare exact results. Use `rowsort` for row bags, `ORDER BY` for promised order, `typeof` or `DESCRIBE` for host types, and stable `QW_*` diagnostics for expected errors. [Command](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/test/sqlite/sqllogic_command.cpp) · [Result checks](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/test/sqlite/result_helper.cpp) |
| `RegisterSqllogictests` and `TryReadExactSQLLogicTestFilter` | Supply an exact absolute file list with `-f`. Check `--emit-test-events` terminal records and per-file assertion-block counts. [Registration](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/test/sqlite/test_sqllogictest.cpp) · [Unittest entry point](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/test/unittest.cpp) |
| Upstream extension load and global option tests | `LOAD` the packaged artifact and use the same DuckDB instance for cross-connection reset and prepared-read invalidation. [V2 load test](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/test/extension/load_extension_capi_v2.test) · [Global option test](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/test/extension/reset_global_extension_option.test) |

QuackWeave anchors are `GraphQuery::bind` for dialect and generation, `Scan::next` for the Arrow schema and cursor, and Ladybug `Session::execute` for native admission, prepare and dispatch. The existing acceptances and Rust tests provide the baseline case inventory. [DuckDB SQLLogicTest guide](https://duckdb.org/docs/stable/dev/sqllogictest/intro)

## Runner and fixture lifecycle

```mermaid
flowchart LR
    Build[Versioned artifact] --> Version[Verify DuckDB version]
    Version --> Seed[Seed isolated Ladybug graph]
    Seed --> Before[Observe initial graph]
    Before --> Run[Run exact SQLLogicTest manifest]
    Run --> After[Observe graph and package]
    After --> Audit[Assert facts in fixture_audit.test]
    Audit --> Evidence[Validate acceptance evidence]
```

- The launcher creates one scratch root per profile and uses [read_graph.cypher](test/stabilization/fixtures/read_graph.cypher) to provision duplicate edges, a self-loop, nullable properties, Unicode and punctuation, and 5,001 linked sample rows. It creates a corrupt graph and reserves a path for the denied export.
- The runner stages the extension under a path with spaces and removes loader environment overrides. The product artifact loads into the pinned DuckDB engine and uses the real Ladybug file.
- An independent Ladybug CLI session captures nodes, relationships and catalog names before and after the suite. SQL compares the canonical snapshots, including multiplicity. A second fixture captures footer fields, exported symbols, imports, digest pairs and export-file presence as CSV. SQL verifies those raw facts using the built-in CSV reader.
- The launcher checks nonzero exit, missing or duplicate terminal records, and per-file block counts. It includes the timeout case explicitly. Expected SQL values, diagnostics, schema and fixture verdicts stay in the `.test` files.

## Reference suite and coverage

| File | Case logic |
|---|---|
| [reads.test](test/stabilization/sqllogic/reads.test) | Fixed and variable-length Cypher, GQL/Cypher bag parity, case-insensitive dialect and alias, self-loops, nullable scalar values and types, all comparison operators, FILTER forms, Unicode/quotes, joins and empty results. |
| [bind.test](test/stabilization/sqllogic/bind.test) | Constant arguments, invalid and reset config, stale prepared reads, SQL schema arguments, GQL source diagnostics and closed subset, variable/alias scope, no fallback, and bind versus execute against a corrupt graph. |
| [admission.test](test/stabilization/sqllogic/admission.test) | Reject invalid native input, writes, batches, routines and export. Prove a lowered GQL query with an unknown property fails native preparation. Verify graph sentinel rows after rejects. |
| [scan.test](test/stabilization/sqllogic/scan.test) | Arrow name/type/order errors, nested output rejection, 5,001-row checksum, early close, connection recovery and batch quota reset. |
| [limits.test](test/stabilization/sqllogic/limits.test) | Both sides of existing 16 KiB, 256-token and 16-level input envelopes, signed integer bounds, and invalid Unicode scalars. |
| [host_access.test](test/stabilization/sqllogic/host_access.test) · [timeout.test_slow](test/stabilization/sqllogic/timeout.test_slow) | Host access recheck, native timeout and same-connection recovery. |
| [artifact.test](test/stabilization/sqllogic/artifact.test) · [native_surface.test](test/stabilization/sqllogic/native_surface.test) | The relocated package includes its GQL parser. The same package joins `GRAPH_TABLE` to bound `qw_query` results and supports standalone reads. |
| [fixture_audit.test](test/stabilization/sqllogic/fixture_audit.test) | Independent graph and export checks, exact ABI footer, initializer, imports, and the three pinned GQL notices. |

The native debug and release runs use the same case manifest. The runner confirms all assertion blocks in each selected file completed.

## Native observation fixtures

The fixture producers collect raw observations. [observer_audit.test](test/stabilization/sqllogic/observer_audit.test) asserts the exact invocation inventory, contiguous sequence numbers, expected event order, typed values and outcomes.

| Case | Observations and SQL oracle |
|---|---|
| `observer.parser` | Native typed literals, raw UTF-8 input, Unicode byte spans and source diagnostic position. SQL validates the span slices against the original bytes and recomputes code-point columns. |
| `observer.admission` | Actual Ladybug AST, C prepare, prepared read-only check, execute, Arrow pull and connection release. AST rejection has zero prepares and executes. A failed native prepare has zero executes. |
| `observer.schema` | Unsupported native Arrow schema observes one execute, zero pulls and one connection release. The core scan also checks the declared schema before returning a batch. |
| `observer.lifecycle` | The real core scan state machine with controlled readers covers EOF, zero rows, early close, schema rejection and first/late batch errors. The pinned DuckDB streaming client independently observes actual packaged row delivery. |
| `observer.quota` | The measured native owned-byte threshold B accepts a batch at B and rejects B−1. SQL checks the reported bounds and the outcomes. |
| `observer.failure` | Test-only fault injection exercises native Arrow, execution and timeout error classification and release. The earlier successful returned Arrow batch remains explicit in the native trace. |
| `observer.policy` | Compare `QW_ACCESS` after a post-bind path swap with `QW_OPEN` after binding directly to the same corrupt graph. Both deliver zero rows. Restore the path and confirm the same prepared statement delivers rows and both fixture hashes are unchanged. |

Each observation is keyed by `fixture_version, case_id, invocation_id, sequence`. All 23 expected invocations have a start and end. A missing invocation fails even when its oracle expects zero calls. The native `release` event refers to connection destruction. Core `batch_ready` is a batch returned by the scan. Packaged `chunk` records are actual rows delivered through DuckDB's streaming C API.

The release package is exercised separately from the Rust fault instrumentation. The observer report hash must match the release report hash. Real timeout behavior is covered by `timeout.test_slow`. Synthetic fault reports identify the injected event, and runner failures cannot satisfy structured SQL error assertions.

[Collector](test/stabilization/run_observer_audit.py) · [Core fixtures](crates/core/src/qw_query/tests/observer.rs) · [Native C observers](crates/native/src/ladybug/ffi/observed.rs) · [Packaged client fixture](test/stabilization/capture_host_observations.py)

## Reproduce the pinned gate

Use the DuckDB checkout at `8d78114e90a53823803660bf821f6073ac87da8c`. Set the explicit version in CMake so an exact shallow checkout reports the release version. The native runner and shared library must come from the same build. Both the SQLLogicTest pin probe and the fixture C API check enforce `v2.0.1` and source ID `8d78114e90`.

```sh
cmake -S /path/to/duckdb-v2.0.1 -B /path/to/duckdb-build \
  -DCMAKE_BUILD_TYPE=Release -DDUCKDB_EXPLICIT_VERSION=v2.0.1 \
  -DBUILD_SHELL=OFF -DBUILD_UNITTESTS=ON -DENABLE_UNITTEST_CPP_TESTS=OFF
cmake --build /path/to/duckdb-build --target unittest --parallel 2

make native-deps
make native-stabilization-test \
  DUCKDB_UNITTEST=/path/to/duckdb-build/test/unittest \
  DUCKDB_LIB=/path/to/duckdb-build/src/libduckdb.dylib \
  LBUG=/path/to/lbug-v0.22.3
```

The gate writes the three run logs, raw observer CSV and `stabilization-coverage.json` to `build/stabilization-reports` by default. The [ARM64 CI workflow](.github/workflows/CI.yml) prepares the pinned DuckDB runner and runs this same Make target.

## Release acceptance gate

A complete mapping, successful SQL execution, successful fixture audit and complete native-observer evidence are required for acceptance. The diagram checks this decision logic. Runtime test reports and fixtures supply the inputs.

In [ ]:
flowchart TD
    SPEC["`@spec QW-STABILIZATION-ACCEPTANCE
@type Status = enum[unmapped, sql_failed_or_missing, fixture_failed_or_missing, observer_pending, accepted]
@input mappings_complete: Bool
@input sql_evidence_complete: Bool
@input fixture_evidence_complete: Bool
@input observer_evidence_complete: Bool
@output status: Status
@output release_accepted: Bool
@requires DOMAIN: true`"]
    A["`@branch UNMAPPED
@when not mappings_complete
@ensures STATUS_A: status = unmapped
@ensures GATE_A: release_accepted = false`"]
    B["`@branch SQL_GAP
@when mappings_complete and not sql_evidence_complete
@ensures STATUS_B: status = sql_failed_or_missing
@ensures GATE_B: release_accepted = false`"]
    C["`@branch FIXTURE_GAP
@when mappings_complete and sql_evidence_complete and not fixture_evidence_complete
@ensures STATUS_C: status = fixture_failed_or_missing
@ensures GATE_C: release_accepted = false`"]
    D["`@branch OBSERVER_GAP
@when mappings_complete and sql_evidence_complete and fixture_evidence_complete and not observer_evidence_complete
@ensures STATUS_D: status = observer_pending
@ensures GATE_D: release_accepted = false`"]
    E["`@branch ACCEPTED
@when mappings_complete and sql_evidence_complete and fixture_evidence_complete and observer_evidence_complete
@ensures STATUS_E: status = accepted
@ensures GATE_E: release_accepted = true`"]
    VERIFY["`@verify CONSISTENT: witness consistency
@verify DETERMINISTIC: prove determinism
@verify COVERED: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify EACH_STATUS: witness each status`"]
    SPEC --> A --> VERIFY
    SPEC --> B --> VERIFY
    SPEC --> C --> VERIFY
    SPEC --> D --> VERIFY
    SPEC --> E --> VERIFY

## Stabilization implementation

| Step | Result |
|---|---|
| S1 · Baseline | Native SQLLogicTest package suite exercises the native debug and release packages. Independent Ladybug snapshots, relocation and package metadata are asserted in SQL. |
| S2 · Observer evidence | Seven groups exercise 23 invocations with real native boundaries, controlled faults and actual packaged streaming. SQL rejects missing invocations, reordered read-only guards and missing release events. |
| S3 · CI gate | The ARM64 workflow builds the pinned DuckDB runner and library together. The Make gate records exact SQL source hashes, artifact hashes and ordered runner phases. |
| S4 · Coverage cutover | The current gate verifies 142 distinct groups across 277 case/run pairs. The standalone acceptance checks are part of `make native-test`. |

The gate requires the full declared inventory and fresh source evidence. A failure reaches its specific diagnostic or observer predicate. A missing, stale, skipped or incomplete report fails the coverage check. [Case manifest](test/stabilization/case_manifest.csv) · [Coverage checker](test/stabilization/check_coverage.py)

## Baseline JEV Compile and Z3 coverage evidence

Coverage uses the catalog's `data_integrity.foreign_key` rule in verification mode. Each required pair of profile and exact case set must exist in the evidence relation. Actual plan evidence came from case markers in the suite and the observer design. Actual runtime evidence came from terminal test records with exact block counts.

| Check | JEV gate and weakest confidence | Z3 | Persisted result |
|---|---|---|---|
| Required suite and observer plan | open · 0.97 | `sat` | `sol_eaae3c8b47e94e3d` |
| Release graph audit removed from the plan | open · 0.98 | `unsat` | `sol_e644b244e0e849c6` |
| Observed runtime for the shipped reference | open · 0.97 | `sat` | `sol_01f420cc14674b0b` |
| Observed runtime against the full stabilization gate | open · 0.88 | `unsat` | `sol_e29286db658c40a1` |

JEV version `jev-1.14.0` proposed one binding per check. The executed rule, mode, subjects and normalized facts matched the candidates. Each solver receipt was reloaded. The first model had 90 required case/profile pairs. The current runtime satisfies all 83 reference-suite pairs. At baseline, the seven native observer pairs still required evidence. The implemented gate is verified below.

### Replay the finite set check

The catalog caps solver variables, so the case lists are encoded as exact integer masks per profile. Case number `n` is bit `n - 1`. Each profile has one active row and every case ID is unique within it. The mask is injective over the 49 declared case IDs. Bit masks preserve the entire membership set.

| Profile | Profile code |
|---|---:|
| Debug | 1 |
| Release | 2 |
| Cypher-only | 3 |
| Native observer | 4 |

Use `solve_rule_spec` with `data_integrity.foreign_key`. Encode each table below as one row per profile in the `required_evidence` or `case_evidence` relation, with integer columns `profile` and `case_mask`. Set both cells present. Use the two columns in that order in the `coverage` foreign key from required to evidence. Send one subject `coverage` with empty parameters/unknowns to Jev. Execute the vetted candidate with `solve_rules`, persistence enabled.

```json
{
  "plan": {
    "required": {
      "debug": 4397979402239,
      "release": 4397979402239,
      "mvp1": 67108864,
      "observed": 558551906910208
    },
    "evidence": {
      "debug": 4397979402239,
      "release": 4397979402239,
      "mvp1": 67108864,
      "observed": 558551906910208
    },
    "expected_status": "sat",
    "receipt": "sol_eaae3c8b47e94e3d"
  },
  "missing": {
    "required": {
      "debug": 4397979402239,
      "release": 4397979402239,
      "mvp1": 67108864,
      "observed": 558551906910208
    },
    "evidence": {
      "debug": 4397979402239,
      "release": 4397710966783,
      "mvp1": 67108864,
      "observed": 558551906910208
    },
    "expected_status": "unsat",
    "receipt": "sol_e644b244e0e849c6"
  },
  "runtime": {
    "required": {
      "debug": 4397979402239,
      "release": 4397979402239,
      "mvp1": 67108864
    },
    "evidence": {
      "debug": 4397979402239,
      "release": 4397979402239,
      "mvp1": 67108864
    },
    "expected_status": "sat",
    "receipt": "sol_01f420cc14674b0b"
  },
  "gap": {
    "required": {
      "debug": 4397979402239,
      "release": 4397979402239,
      "mvp1": 67108864,
      "observed": 558551906910208
    },
    "evidence": {
      "debug": 4397979402239,
      "release": 4397979402239,
      "mvp1": 67108864
    },
    "expected_status": "unsat",
    "receipt": "sol_e29286db658c40a1"
  }
}
```

## Historical case inventory

See the [current case manifest](test/stabilization/case_manifest.csv) for the active release gates.

| Case number | Required case | Assertion source |
|---:|---|---|
| 1 | `read.setup` | [reads.test](test/stabilization/sqllogic/reads.test#L9) |
| 2 | `read.cypher` | [reads.test](test/stabilization/sqllogic/reads.test#L16) |
| 3 | `read.gql` | [reads.test](test/stabilization/sqllogic/reads.test#L30) |
| 4 | `read.parity` | [reads.test](test/stabilization/sqllogic/reads.test#L37) |
| 5 | `read.filter` | [reads.test](test/stabilization/sqllogic/reads.test#L50) |
| 6 | `read.self` | [reads.test](test/stabilization/sqllogic/reads.test#L122) |
| 7 | `read.scalar` | [reads.test](test/stabilization/sqllogic/reads.test#L128) |
| 8 | `read.join` | [reads.test](test/stabilization/sqllogic/reads.test#L135) |
| 9 | `read.empty` | [reads.test](test/stabilization/sqllogic/reads.test#L142) |
| 10 | `read.literal` | [reads.test](test/stabilization/sqllogic/reads.test#L153) |
| 11 | `bind.config` | [bind.test](test/stabilization/sqllogic/bind.test#L13) |
| 12 | `bind.generation` | [bind.test](test/stabilization/sqllogic/bind.test#L112) |
| 13 | `bind.parse` | [bind.test](test/stabilization/sqllogic/bind.test#L127) |
| 14 | `bind.subset` | [bind.test](test/stabilization/sqllogic/bind.test#L139) |
| 15 | `bind.scope` | [bind.test](test/stabilization/sqllogic/bind.test#L221) |
| 16 | `bind.plan` | [bind.test](test/stabilization/sqllogic/bind.test#L242) |
| 17 | `admission.setup` | [admission.test](test/stabilization/sqllogic/admission.test#L11) |
| 18 | `admission.cypher` | [admission.test](test/stabilization/sqllogic/admission.test#L18) |
| 19 | `admission.prepare` | [admission.test](test/stabilization/sqllogic/admission.test#L88) |
| 20 | `admission.guard` | [admission.test](test/stabilization/sqllogic/admission.test#L94) |
| 21 | `scan.setup` | [scan.test](test/stabilization/sqllogic/scan.test#L9) |
| 22 | `scan.schema` | [scan.test](test/stabilization/sqllogic/scan.test#L16) |
| 23 | `scan.multibatch` | [scan.test](test/stabilization/sqllogic/scan.test#L37) |
| 24 | `scan.budget` | [scan.test](test/stabilization/sqllogic/scan.test#L54) |
| 25 | `host.disable` | [host_access.test](test/stabilization/sqllogic/host_access.test#L9) |
| 26 | `scan.timeout` | [timeout.test_slow](test/stabilization/sqllogic/timeout.test_slow#L9) |
| 28 | `artifact.load` | [artifact.test](test/stabilization/sqllogic/artifact.test#L9) |
| 29 | `fixture.graph` | [fixture_audit.test](test/stabilization/sqllogic/fixture_audit.test#L15) |
| 30 | `fixture.export` | [fixture_audit.test](test/stabilization/sqllogic/fixture_audit.test#L22) |
| 31 | `fixture.footer` | [fixture_audit.test](test/stabilization/sqllogic/fixture_audit.test#L28) |
| 32 | `fixture.exports` | [fixture_audit.test](test/stabilization/sqllogic/fixture_audit.test#L36) |
| 33 | `fixture.notices` | [fixture_audit.test](test/stabilization/sqllogic/fixture_audit.test#L53) |
| 34 | `limits.bytes` | [limits.test](test/stabilization/sqllogic/limits.test#L15) |
| 35 | `limits.tokens` | [limits.test](test/stabilization/sqllogic/limits.test#L24) |
| 36 | `limits.depth` | [limits.test](test/stabilization/sqllogic/limits.test#L33) |
| 37 | `limits.literal` | [limits.test](test/stabilization/sqllogic/limits.test#L44) |
| 38 | `bind.config_reject` | [bind.test](test/stabilization/sqllogic/bind.test#L66) |
| 39 | `read.comparison` | [reads.test](test/stabilization/sqllogic/reads.test#L95) |
| 40 | `bind.shape` | [bind.test](test/stabilization/sqllogic/bind.test#L170) |
| 41 | `admission.empty_controls` | [admission.test](test/stabilization/sqllogic/admission.test#L77) |
| 42 | `fixture.setup` | [fixture_audit.test](test/stabilization/sqllogic/fixture_audit.test#L11) |
| 43 | `observer.parser` | [observer_audit.test](test/stabilization/sqllogic/observer_audit.test#L198) |
| 44 | `observer.admission` | [observer_audit.test](test/stabilization/sqllogic/observer_audit.test#L74) |
| 45 | `observer.schema` | [observer_audit.test](test/stabilization/sqllogic/observer_audit.test#L118) |
| 46 | `observer.lifecycle` | [observer_audit.test](test/stabilization/sqllogic/observer_audit.test#L245) |
| 47 | `observer.quota` | [observer_audit.test](test/stabilization/sqllogic/observer_audit.test#L130) |
| 48 | `observer.failure` | [observer_audit.test](test/stabilization/sqllogic/observer_audit.test#L151) |
| 49 | `observer.policy` | [observer_audit.test](test/stabilization/sqllogic/observer_audit.test#L266) |

An expected case maps to a validated file and profile, and a file maps to a terminal runner record with all blocks passing. The observer cases use raw fixture records with required start/end evidence. The model validates this mapping and the missing-evidence controls. SQLLogicTest and the observation fixtures provide behavioral evidence.

## Runtime negative controls

The pinned runner rejected a changed fixture snapshot. The unified surface regression also returned a parser error at `MATCH` when run against the older Cypher-only artifact. The same SQL passed on both unified native packages.

[Unified surface regression](test/stabilization/sqllogic/native_surface.test) · [Fixture audit](test/stabilization/sqllogic/fixture_audit.test)

## Verification outputs

Run `make native-stabilization-test` to record the debug, release and observer logs. The coverage checker verifies the SQL file hashes, the observer package identity, and all ordered runner events. The capability manifest also records passing coverage for RelateDB, the Neo4j TLS fixture, and the ADBC client.

[Capability evidence manifest](docs/evidence.json) · [Coverage checker](test/stabilization/check_coverage.py)